In [ ]:
from pathlib import Path

%pip install --quiet --disable-pip-version-check requests==2.34.2
%pip install --quiet --disable-pip-version-check pandas==3.0.6
%pip install --quiet --disable-pip-version-check numpy==2.5.3
%pip install --quiet --disable-pip-version-check scikit-learn==1.9.1
%pip install --quiet --disable-pip-version-check matplotlib==3.11.2
%pip install --quiet --disable-pip-version-check pandas_market_calendars==5.4.0
%pip install --quiet --disable-pip-version-check prophet==1.4.0
import logging

logging.basicConfig(level=logging.WARNING)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pandas_market_calendars as mcal
import requests
import os
from getpass import getpass
from urllib.parse import quote
from sklearn.metrics import accuracy_score, balanced_accuracy_score, ConfusionMatrixDisplay
from IPython.display import display

plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

from prophet import Prophet

API_KEY = os.environ.get("EULERPOOL_API_KEY", "").strip()

In [ ]:
SYMBOL = "BTC-USD"
START = "2023-10-01"
END = "2026-10-04"
TEST_DAYS = 60
TRAIN_DAYS = 365
SEED = 42
CALENDAR = "24/7" if SYMBOL.endswith("-USD") else "NYSE"

DATA = Path("data")
OUTPUT = Path("results") / "Eulerpool" / SYMBOL
DATA.mkdir(exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)

MODEL = "Prophet"

In [ ]:
def eulerpool_request(path, params=None):
    global API_KEY
    if not API_KEY:
        API_KEY = getpass("Eulerpool API key: ").strip()
    if not API_KEY:
        raise ValueError("An Eulerpool API key is required.")
    try:
        response = requests.get(
            f"https://api.eulerpool.com/api/1/{path}", params=params,
            headers={"Authorization": f"Bearer {API_KEY}", "Accept": "application/json"},
            timeout=(3, 15), allow_redirects=False,
        )
    except requests.RequestException:
        raise ValueError("Eulerpool could not be reached. Try again later.") from None
    if response.status_code in (401, 403):
        raise ValueError("Check your Eulerpool API key and data access.")
    if response.status_code == 429:
        raise ValueError("Eulerpool's request limit was reached. Try again later.")
    if response.status_code != 200:
        raise ValueError(f"Eulerpool could not return this data (status {response.status_code}).")
    try:
        return response.json()
    except ValueError:
        raise ValueError("Eulerpool returned an unreadable response.") from None


def eulerpool_prices(symbol, start, end):
    crypto = symbol.endswith("-USD")
    identifier = quote(symbol.removesuffix("-USD") if crypto else symbol, safe="")
    category = "crypto" if crypto else "equity"
    profile = eulerpool_request(f"{category}/profile/{identifier}")
    if not isinstance(profile, dict) or str(profile.get("currency", "")).upper() != "USD":
        raise ValueError("This experiment needs an asset with prices quoted in USD.")
    start = pd.Timestamp(start).normalize().tz_localize("UTC")
    end = min(pd.Timestamp(end).normalize().tz_localize("UTC"), pd.Timestamp.now(tz="UTC").normalize())
    if start >= end:
        raise ValueError("The end date must be after the start date.")
    if crypto:
        data = eulerpool_request(f"crypto/quotes/{identifier}", {
            "startdate": int(start.timestamp() * 1000), "enddate": int(end.timestamp() * 1000),
        })
        try:
            timestamps = [row["timestamp"] for row in data]
            values = [row["price"] for row in data]
        except (KeyError, TypeError):
            raise ValueError("Eulerpool returned unreadable crypto prices.") from None
        unit = "ms"
    else:
        data = eulerpool_request(f"charting/ohlcv/{identifier}", {
            "resolution": "D", "from": int(start.timestamp()), "to": int(end.timestamp()),
        })
        if not isinstance(data, dict):
            raise ValueError("Eulerpool returned unreadable stock prices.")
        timestamps, values, unit = data.get("t", []), data.get("c", []), "s"
    if not timestamps or len(timestamps) != len(values):
        raise ValueError("Eulerpool did not return a complete price response.")
    try:
        dates = pd.to_datetime(timestamps, unit=unit, utc=True, errors="raise")
        prices = pd.Series(pd.to_numeric(values, errors="raise"), index=dates, dtype=float).sort_index()
    except (ValueError, TypeError, OverflowError):
        raise ValueError("Eulerpool returned invalid prices or dates.") from None
    prices = prices.loc[(prices.index >= start) & (prices.index < end)]
    if prices.empty or prices.index.hasnans or not np.isfinite(prices).all() or prices.le(0).any():
        raise ValueError("Eulerpool returned missing or invalid prices.")
    prices.index = prices.index.tz_localize(None).normalize()
    prices = prices.groupby(level=0).last()
    if not crypto:
        prices = prices.loc[prices.index.dayofweek < 5]
    return prices.rename("Price").rename_axis("Date")

In [ ]:
path = DATA / f"Eulerpool_{SYMBOL}_{START}_{END}.csv"
if not path.exists():
    eulerpool_prices(SYMBOL, START, END).to_csv(path)

prices = pd.read_csv(path, index_col="Date", parse_dates=True)["Price"]
cutoff = min(pd.Timestamp(END), pd.Timestamp.now(tz="UTC").normalize().tz_localize(None))
prices = prices.loc[(prices.index >= pd.Timestamp(START)) & (prices.index < cutoff)]
if not prices.index.is_unique or not prices.index.is_monotonic_increasing:
    raise ValueError("Price dates must be unique and ordered.")
if prices.empty or not np.isfinite(prices).all() or prices.le(0).any():
    raise ValueError("Prices must be positive and complete.")
calendar = mcal.get_calendar(CALENDAR)
expected = calendar.valid_days(pd.Timestamp(START), cutoff - pd.Timedelta(days=1)).tz_localize(None)
if not prices.index.equals(expected):
    missing = expected.difference(prices.index)
    raise ValueError(
        f"Eulerpool has {len(missing)} missing expected price dates, or the calendar does not match. "
        "Use complete daily data before running the experiment. Missing prices are not filled."
    )
if len(prices) < TRAIN_DAYS + TEST_DAYS + 1:
    raise ValueError("There is not enough price history.")
test_dates = prices.index[-TEST_DAYS:]
display(pd.Series({
    "Asset": SYMBOL,
    "Source": "Eulerpool: https://eulerpool.com/developers",
    "Prices": f"{prices.index[0].date()} to {prices.index[-1].date()}",
    "Target": "Next daily price: Up or Down",
    "Test": f"{test_dates[0].date()} to {test_dates[-1].date()}",
    "Method": "Refit before each prediction using the previous 366 daily prices",
    "Price basis": "Last available observation per UTC day; stock adjustments are not verified",
}))

In [ ]:
def majority_trend(history):
    changes = history.pct_change().dropna()
    changes = changes.loc[changes.ne(0)]
    return "Up" if changes.gt(0).mean() >= 0.5 else "Down"

In [ ]:
def predict(history, target_date):
    training = pd.DataFrame({"ds": history.index, "y": history.to_numpy()})
    model = Prophet(
        yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False,
        n_changepoints=10, changepoint_prior_scale=0.05, uncertainty_samples=0,
    )
    model.fit(training, seed=SEED, algorithm="Newton")
    forecast = float(model.predict(pd.DataFrame({"ds": [target_date]}))["yhat"].iloc[0])
    change = forecast / float(history.iloc[-1]) - 1
    trend = "Up" if change > 0 else "Down" if change < 0 else "Flat"
    return {"Predicted trend": trend, "Forecast change %": change * 100}

In [ ]:
rows = []
for target_date in test_dates:
    history = prices.loc[prices.index < target_date].tail(TRAIN_DAYS + 1)
    prediction = predict(history, target_date)
    rows.append({
        "Date": target_date,
        "Training through": history.index[-1],
        "Last price": float(history.iloc[-1]),
        **prediction,
        "Baseline trend": majority_trend(history),
    })

predictions = pd.DataFrame(rows).set_index("Date")
predictions["Actual price"] = prices.loc[test_dates]
predictions["Actual change %"] = (predictions["Actual price"] / predictions["Last price"] - 1) * 100
predictions["Actual trend"] = np.select(
    [predictions["Actual change %"] > 0, predictions["Actual change %"] < 0],
    ["Up", "Down"], default="Flat",
)
predictions["Correct"] = predictions["Predicted trend"] == predictions["Actual trend"]
display(predictions[["Predicted trend", "Actual trend", "Correct"]].tail(10))

In [ ]:
scored = predictions.loc[predictions["Actual trend"] != "Flat"]
if not (not scored.empty):
    raise ValueError('No up or down moves were found in the evaluation period.')
scores = []
for name, column in [(MODEL, "Predicted trend"), ("Majority direction", "Baseline trend")]:
    scores.append({
        "Model": name,
        "Accuracy %": accuracy_score(scored["Actual trend"], scored[column]) * 100,
        "Balanced accuracy %": balanced_accuracy_score(scored["Actual trend"], scored[column]) * 100,
        "Test days": len(scored),
        "Flat days excluded": len(predictions) - len(scored),
    })
metrics = pd.DataFrame(scores).set_index("Model")
display(metrics.round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
ConfusionMatrixDisplay.from_predictions(
    scored["Actual trend"], scored["Predicted trend"], labels=["Down", "Up"],
    cmap="Greens", colorbar=False, ax=axes[0],
)
axes[0].set_title(f"{MODEL}: next-day direction")
metrics["Accuracy %"].plot.bar(ax=axes[1], color=["#266650", "#9aa7a0"], rot=0)
axes[1].set_ylim(0, 100)
axes[1].set_xlabel("")
axes[1].set_ylabel("Correct predictions (%)")
axes[1].set_title("Test accuracy")
fig.suptitle(f"{SYMBOL} | {test_dates[0].date()} to {test_dates[-1].date()}")
fig.savefig(OUTPUT / "test_results.png", dpi=160)
plt.show()

In [ ]:
future_dates = calendar.valid_days(
    prices.index[-1] + pd.Timedelta(days=1),
    prices.index[-1] + pd.Timedelta(days=14),
).tz_localize(None)
target_date = future_dates[0]
history = prices.tail(TRAIN_DAYS + 1)
latest = pd.DataFrame([{
    "Asset": SYMBOL, "Model": MODEL, "Date": target_date,
    "Training through": history.index[-1], **predict(history, target_date),
}])
display(latest)

In [ ]:
predictions.to_csv(OUTPUT / "test_predictions.csv")
metrics.to_csv(OUTPUT / "metrics.csv")
latest.to_csv(OUTPUT / "latest_prediction.csv", index=False)

This experiment now uses Eulerpool and needs to be rerun before we can compare Prophet with the majority-direction rule. We have cleared the earlier output because those results came from Yahoo Finance. If Eulerpool has missing daily prices, the notebook will stop before fitting the model. Once we have a complete history, we can compare Prophet and LSTM over the same dates and see whether either model adds anything useful.